# Lisa: viie mudeli võrdluse tulemused

Ridge, juhumets, juhuslike lõigetega mets, GBR ning HistGBR katsetatakse samade tunnusevaadete ja ajaliste jaotustega. Iga perekond saab neli kandidaati. Parima meetodi järeldus peab nimetama andmestiku, perioodi, kandidaadid ning ebakindluse.

In [ ]:
from pathlib import Path
from functools import partial
import sys
import json
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "run_pipeline.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.common import latest_run, read_json
from src.reporting import show_table, show_figure, show_run
from src.expanded_reporting import display_table

EXPANDED = latest_run("latest_expanded_run.json")
captions = read_json(EXPANDED / "reports/figure_interpretations.json")
conclusion = read_json(EXPANDED / "reports/conclusion.json")
expanded_table = partial(show_table, EXPANDED, formatter=display_table)


def expanded_figure(name):
    show_figure(EXPANDED, "figures/" + name, captions[name], width=1100)


show_run(EXPANDED, expanded=True)

In [ ]:
settings = read_json(EXPANDED / "configs/expanded_study.json")["models"]
from src.expanded_reporting import MODEL_NAMES

parameter_table = pd.DataFrame(
    [
        {
            "Mudel": MODEL_NAMES[name],
            "Seadistus": number,
            "Parameetrid": json.dumps(values, ensure_ascii=False, sort_keys=True),
        }
        for name, options in settings.items()
        for number, values in enumerate(options)
    ]
)
display(Markdown(display_table(parameter_table)))
display(
    Markdown(
        "Tabel näitab enne katseid fikseeritud kümmet parameetriseadistust. Iga seadistust võrreldi kahel ühisel tunnusevaatel. See on piiratud otsing, mille tulemust ei nimetata perekonna absoluutseks optimaalseks seadistuseks; HistGBR-i juhuslik varajane peatamine on keelatud."
    )
)

In [ ]:
expanded_table(
    "selection/model_ranking.csv",
    [
        "model",
        "feature_set",
        "parameter_id",
        "MAE",
        "RMSE",
        "fold_MAE_std",
        "n_features",
        "fit_seconds",
    ],
)
display(
    Markdown(
        "Tabel sisaldab kõiki 20 mudeli/tunnuse/seadistuse kandidaati, ka halvemaid. MAE on 2017–2018 vaatluste arvuga kaalutud skoor. Kuna see on kasutatud parameetrivalikuks, ei ole see erapooletu lõppvea hinnang."
    )
)
expanded_figure("04_model_feature_heatmap.png")
expanded_figure("06_accuracy_cost.png")

In [ ]:
expanded_table(
    "evaluation/performance.csv",
    [
        "period",
        "model",
        "feature_set",
        "MAE",
        "RMSE",
        "MAPE",
        "R2",
        "bias",
        "n_features",
        "fit_predict_seconds",
    ],
)
display(
    Markdown(
        "Tabel näitab ette fikseeritud finalistide hilisemat tulemust samadel vaatlustel. Persistence ja eelmise päeva prognoos on lihtsad võrdluspunktid. 2019 tulemust kasutatakse eelneva valiku kontrolliks; 2020 kirjeldab varasemale andmestikule tagasivaatelist ülekantavust."
    )
)
expanded_figure("05_model_temporal_stability.png")

In [ ]:
expanded_table(
    "evaluation/paired_intervals.csv",
    ["period", "A", "B", "difference", "adjusted_low", "adjusted_high", "block_hours"],
)
display(
    Markdown(
        "Negatiivne A miinus B erinevus soosib valikuvõitjat A. Nelja võrdluse Bonferroni tasemega intervalli nulli hõlmamine ei kinnita paremust vastava konkurendi ees. 168 tundi on peamine plokipikkus, 24 tundi tundlikkuskontroll; eri seadistuste kõiki võrdlusi ei käsitleta ühe korrigeeritud testiperekonnana."
    )
)
expanded_figure("07_paired_uncertainty.png")
expanded_figure("08_scenario_comparison.png")

In [ ]:
display(Markdown(conclusion["text"]))
expanded_table(
    "evaluation/all_pairwise.csv",
    ["period", "A", "B", "difference", "adjusted_low", "adjusted_high", "block_hours"],
)
display(
    Markdown(
        "Kõigi kümne mudelipaari võrdlus on uuriv järelanalüüs. Vahemikud on Bonferroni parandusega ühe perioodi ja plokipikkuse kohta. Negatiivne A−B vahe soosib mudelit A; nulli hõlmav vahemik ei kinnita paremust."
    )
)

Täpsuse poolest parem mudeliperekond ei tõenda XAI-põhise tunnusevaliku paremust. Algse uurimuse GBR jääb diagnoosi võrdlusmudeliks. Uus pime ajaperiood oleks vajalik tugevamaks sõltumatuks kinnituseks.